<a href="https://colab.research.google.com/github/tabaraei/MS-Thesis/blob/main/Depression_Detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

TODO General:
- Understand the division between train/test sets
- Obtain and save the labels for further usage

TODO Text (`XXX_TRANSCRIPT.csv`):
- Look for a proper model first
- See what the model requires as input (lemmatization, tokenization, vectorization)
- Loop over all the text files:
    - Prepare a standard text extracted from the CSV file
    - Normalize if needed
    - Feed the text to model
    - Append the extracted feature for each text to a feature set
- Save the feature embeddings for further usage

TODO Audio (`XXX_AUDIO.wav` at 16KHz):
- Look for a proper model first
- See what the model requires as input
- Loop over all the audio files:
    - Load the wav file
    - Perform normalization if needed
    - Feed the audio to model
    - Append the extracted feature for each audio to a feature set
- Save the feature embeddings for further usage

In [1]:
%%capture

import os
import requests
import zipfile
from io import BytesIO
from google.colab import drive
from tqdm import tqdm
import pandas as pd
import librosa
from IPython.display import Audio

drive.mount('/content/drive')
DRIVE_PATH = '/content/drive/MyDrive/Data'

In [2]:
class DAIC_WoZ:
    def __init__(self, DRIVE_PATH, download: bool=False):
        self.DRIVE_PATH = DRIVE_PATH
        self.DAIC_WoZ_DATA_PATH = f'{DRIVE_PATH}/DAIC-WoZ'
        self.DAIC_WoZ_DOWNLOAD_PATH = 'https://dcapswoz.ict.usc.edu/wwwdaicwoz/'
        self.file_names = {'audio': 'AUDIO.wav', 'text': 'TRANSCRIPT.csv'}
        self.select_sessions()
        if download: self.download_dataset()
        self.prepare_train_dev_test_splits()

    def select_sessions(self):
        all_sessions = numbers = set(range(300, 493))
        excluded_sessions = {342, 394, 398, 460}
        special_sessions = {373, 444, 451, 458, 480, 402}
        self.selected_sessions = all_sessions - excluded_sessions - special_sessions

    def download_dataset(self):
        os.makedirs(self.DAIC_WoZ_DATA_PATH, exist_ok=True)
        zip_files = [f'{session}_P.zip' for session in self.selected_sessions]
        for zip_name in tqdm(zip_files):
            response = requests.get(f'{self.DAIC_WoZ_DOWNLOAD_PATH}/{zip_name}', stream=True)
            response.raise_for_status()
            with zipfile.ZipFile(BytesIO(response.content)) as zf:
                prefix = zip_name[:3]
                for file_name in self.file_names.values():
                    zf.extract(f'{prefix}_{file_name}', path=self.DAIC_WoZ_DATA_PATH)

    def prepare_dataframe(self, row):
        participant_id = int(row['Participant_ID'])
        score_column = 'PHQ8_Score' if 'PHQ8_Score' in row.index else 'PHQ_Score'
        return pd.Series({
            'participant_id': participant_id,
            'audio_path': f"{self.DAIC_WoZ_DATA_PATH}/{participant_id}_{self.file_names['audio']}",
            'text_path': f"{self.DAIC_WoZ_DATA_PATH}/{participant_id}_{self.file_names['text']}",
            'depressed': 1 if row[score_column] >= 10 else 0
        })

    def prepare_train_dev_test_splits(self):
        self.train_df = pd.read_csv(f'{self.DAIC_WoZ_DOWNLOAD_PATH}/train_split_Depression_AVEC2017.csv') \
            .apply(self.prepare_dataframe, axis=1)
        self.dev_df = pd.read_csv(f'{self.DAIC_WoZ_DOWNLOAD_PATH}/dev_split_Depression_AVEC2017.csv') \
            .apply(self.prepare_dataframe, axis=1)
        self.test_df = pd.read_csv(f'{self.DAIC_WoZ_DOWNLOAD_PATH}/full_test_split.csv') \
            .apply(self.prepare_dataframe, axis=1)


DAIC_dataset = DAIC_WoZ(DRIVE_PATH, download=False)

In [3]:
DAIC_dataset.train_df.head()

,participant_id,audio_path,text_path,depressed
0,303,/content/drive/MyDrive/Data/DAIC-WoZ/303_AUDIO...,/content/drive/MyDrive/Data/DAIC-WoZ/303_TRANS...,0
1,304,/content/drive/MyDrive/Data/DAIC-WoZ/304_AUDIO...,/content/drive/MyDrive/Data/DAIC-WoZ/304_TRANS...,0
2,305,/content/drive/MyDrive/Data/DAIC-WoZ/305_AUDIO...,/content/drive/MyDrive/Data/DAIC-WoZ/305_TRANS...,0
3,310,/content/drive/MyDrive/Data/DAIC-WoZ/310_AUDIO...,/content/drive/MyDrive/Data/DAIC-WoZ/310_TRANS...,0
4,312,/content/drive/MyDrive/Data/DAIC-WoZ/312_AUDIO...,/content/drive/MyDrive/Data/DAIC-WoZ/312_TRANS...,0


In [4]:
TEXT_PATH = DAIC_dataset.train_df.loc[0, 'text_path']
sample_text = pd.read_csv(TEXT_PATH, delimiter='\t')
sample_text.head()

,start_time,stop_time,speaker,value
0,26.276,48.696,Ellie,hi i'm ellie thanks for coming in today i was ...
1,49.256,50.406,Ellie,how are you doing today
2,50.686,51.836,Participant,okay how 'bout yourself
3,52.576,54.136,Ellie,i'm great thanks
4,54.816,56.236,Ellie,where are you from originally


In [6]:
AUDIO_PATH = DAIC_dataset.train_df.loc[0, 'audio_path']
waveform, sample_rate = librosa.load(AUDIO_PATH, mono=True, sr=16000)
# Audio(AUDIO_PATH)